## Student Attendance Analytics with Pandas

In [2]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [5]:
df=pd.read_excel("Archive/CSEB_Attendance.xlsx")
df

,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW24CS061,KAVYA KAMATH,Data Science & GenAI - 3nd Yr - CSE - B,12,12,0,0,100%,Safe,✓ Met,2026-10-09,0
1,4MW24CS062,KEERTHANA,Data Science & GenAI - 3nd Yr - CSE - B,12,12,0,0,100%,Safe,✓ Met,2026-10-09,0
2,4MW24CS063,KEERTHI.R.SHETTY,Data Science & GenAI - 3nd Yr - CSE - B,12,12,0,0,100%,Safe,✓ Met,2026-10-09,0
3,4MW24CS064,KHUSHI,Data Science & GenAI - 3nd Yr - CSE - B,12,11,1,0,92%,Safe,✓ Met,2026-10-09,0
4,4MW24CS065,KISHAN KUMAR K,Data Science & GenAI - 3nd Yr - CSE - B,12,11,1,0,92%,Safe,✓ Met,2026-10-09,0
...,...,...,...,...,...,...,...,...,...,...,...,...
60,4MW24CS121,SAHANA M,Data Science & GenAI - 3nd Yr - CSE - B,12,10,2,0,83%,Warning,✓ Met,2026-10-09,0
61,4MW24CS122,SAKSHI,Data Science & GenAI - 3nd Yr - CSE - B,12,11,1,0,92%,Safe,✓ Met,2026-10-09,0
62,4MW25CS404,MANISH M BHANDARY,Data Science & GenAI - 3nd Yr - CSE - B,12,11,1,0,92%,Safe,✓ Met,2026-10-09,0
63,4MW25CS405,MANYA PRARTHANA,Data Science & GenAI - 3nd Yr - CSE - B,12,0,12,0,0%,Critical,36 more,Never,12


#### Section 1: Data Inspection

In [6]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
#-----------------------------Write Solution Code-----------------------------

print("First 10 rows:")
print(df.head(10))

print("\nShape of DataFrame:")
print(df.shape)

print("\nData types of each column:")
print(df.dtypes)


First 10 rows:
      Roll No           Student Name                                  Subject  \
0  4MW24CS061           KAVYA KAMATH  Data Science & GenAI - 3nd Yr - CSE - B   
1  4MW24CS062              KEERTHANA  Data Science & GenAI - 3nd Yr - CSE - B   
2  4MW24CS063       KEERTHI.R.SHETTY  Data Science & GenAI - 3nd Yr - CSE - B   
3  4MW24CS064                 KHUSHI  Data Science & GenAI - 3nd Yr - CSE - B   
4  4MW24CS065         KISHAN KUMAR K  Data Science & GenAI - 3nd Yr - CSE - B   
5  4MW24CS066             KOUSHTUBHA  Data Science & GenAI - 3nd Yr - CSE - B   
6  4MW24CS067                 KRITHI  Data Science & GenAI - 3nd Yr - CSE - B   
7  4MW24CS068               KRITHIKA  Data Science & GenAI - 3nd Yr - CSE - B   
8  4MW24CS069                LAKSHMI  Data Science & GenAI - 3nd Yr - CSE - B   
9  4MW24CS070  LAVANYA SHANKAR MOGER  Data Science & GenAI - 3nd Yr - CSE - B   

   Total Classes  Present  Absent  Excused Attendance %    Status  \
0             12       1

In [7]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """
#-----------------------------Write Solution Code-----------------------------

print("Missing values in each column:")
print(df.isnull().sum())

print("\nColumns containing null values:")
print(df.columns[df.isnull().any()].tolist())

print("\nCount of 'Never' in Last Present Date:")
print(
    df["Last Present Date"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("never")
    .sum()
)


Missing values in each column:
Roll No                 0
Student Name            0
Subject                 0
Total Classes           0
Present                 0
Absent                  0
Excused                 0
Attendance %            0
Status                  0
Classes Needed (75%)    0
Last Present Date       0
Consecutive Absences    0
dtype: int64

Columns containing null values:
[]

Count of 'Never' in Last Present Date:
2


In [8]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
#-----------------------------Write Solution Code-----------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("%", "pct", regex=False)
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Updated column names:")
print(df.columns.tolist())


Updated column names:
['roll_no', 'student_name', 'subject', 'total_classes', 'present', 'absent', 'excused', 'attendance_pct', 'status', 'classes_needed_75pct', 'last_present_date', 'consecutive_absences']


In [9]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
#-----------------------------Write Solution Code-----------------------------

df["attendance_pct"] = (
    df["attendance_pct"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)

df["attendance_pct"] = pd.to_numeric(
    df["attendance_pct"], errors="coerce"
)

df["last_present_date"] = pd.to_datetime(
    df["last_present_date"].replace("Never", pd.NaT),
    errors="coerce"
)

print(df[["attendance_pct", "last_present_date"]].head())
print("\nUpdated data types:")
print(df[["attendance_pct", "last_present_date"]].dtypes)


   attendance_pct last_present_date
0             100        2026-10-09
1             100        2026-10-09
2             100        2026-10-09
3              92        2026-10-09
4              92        2026-10-09

Updated data types:
attendance_pct                int64
last_present_date    datetime64[us]
dtype: object


#### Section 2: Filtering & Selection

In [10]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
#-----------------------------Write Solution Code-----------------------------

critical_students = df[df["status"] == "Critical"]

critical_students = critical_students.sort_values(
    by="attendance_pct",
    ascending=True
)

print(critical_students)


       roll_no     student_name                                  subject  \
10  4MW24CS071       LAXMEESH V  Data Science & GenAI - 3nd Yr - CSE - B   
63  4MW25CS405  MANYA PRARTHANA  Data Science & GenAI - 3nd Yr - CSE - B   
13  4MW24CS074   MAHALINGAPPA C  Data Science & GenAI - 3nd Yr - CSE - B   
5   4MW24CS066       KOUSHTUBHA  Data Science & GenAI - 3nd Yr - CSE - B   
49  4MW24CS110       PRATHWEESH  Data Science & GenAI - 3nd Yr - CSE - B   
46  4MW24CS107  PRATHAM G KUMAR  Data Science & GenAI - 3nd Yr - CSE - B   
35  4MW24CS096    NITHIL SHETTY  Data Science & GenAI - 3nd Yr - CSE - B   
29  4MW24CS090      NIKHIL NAIK  Data Science & GenAI - 3nd Yr - CSE - B   
51  4MW24CS112            RAHUL  Data Science & GenAI - 3nd Yr - CSE - B   

    total_classes  present  absent  excused  attendance_pct    status  \
10             12        0      12        0               0  Critical   
63             12        0      12        0               0  Critical   
13             12   

In [11]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
#-----------------------------Write Solution Code-----------------------------

fully_absent = df[df["present"] == 0]

print(
    fully_absent[
        ["roll_no", "student_name", "consecutive_absences"]
    ]
)


       roll_no     student_name  consecutive_absences
10  4MW24CS071       LAXMEESH V                    12
63  4MW25CS405  MANYA PRARTHANA                    12


In [12]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """
#-----------------------------Write Solution Code-----------------------------

# (a) Display all Safe students using loc
safe_students = df.loc[df["status"] == "Safe"]
print("Safe students:")
print(safe_students)

# (b) First 15 rows and columns 2 to 6 using iloc
print("\nFirst 15 rows and columns 2 to 6:")
print(df.iloc[:15, 1:6])


Safe students:
       roll_no                   student_name  \
0   4MW24CS061                   KAVYA KAMATH   
1   4MW24CS062                      KEERTHANA   
2   4MW24CS063               KEERTHI.R.SHETTY   
3   4MW24CS064                         KHUSHI   
4   4MW24CS065                 KISHAN KUMAR K   
6   4MW24CS067                         KRITHI   
7   4MW24CS068                       KRITHIKA   
8   4MW24CS069                        LAKSHMI   
11  4MW24CS072                      LIKHITA S   
12  4MW24CS073                   LIPIKA NAYAK   
14  4MW24CS075                 MANASA R KULAL   
15  4MW24CS076                    MANOJ KUMAR   
16  4MW24CS077                          MANYA   
17  4MW24CS078                      MANYA B C   
19  4MW24CS080                  MAYUR S KULAL   
20  4MW24CS081                        MEGHANA   
22  4MW24CS083  MOKSHA JEETENDRA MURUDESHWARA   
23  4MW24CS084                   NANDAN NAYAK   
24  4MW24CS085                      NARMADA S   
26  4

In [13]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
#-----------------------------Write Solution Code-----------------------------

students_absent = df[
    df["consecutive_absences"] >= 5
]

print(
    students_absent[
        ["student_name", "status", "last_present_date"]
    ]
)


       student_name    status last_present_date
5        KOUSHTUBHA  Critical        2026-09-18
10       LAXMEESH V  Critical               NaT
35    NITHIL SHETTY  Critical        2026-09-18
46  PRATHAM G KUMAR  Critical        2026-09-18
49       PRATHWEESH  Critical        2026-09-18
63  MANYA PRARTHANA  Critical               NaT


### Section 3: Grouping & Aggregation 

In [14]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""
#-----------------------------Write Solution Code-----------------------------

status_summary = df.groupby("status").agg(
    student_count=("roll_no", "count"),
    average_attendance=("attendance_pct", "mean"),
    max_consecutive_absences=("consecutive_absences", "max")
)

print(status_summary)


          student_count  average_attendance  max_consecutive_absences
status                                                               
Critical              9           17.333333                        12
Safe                 42           96.952381                         0
Warning              14           80.714286                         1


In [15]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""
#-----------------------------Write Solution Code-----------------------------

df["attendance_category"] = np.select(
    [
        df["attendance_pct"] >= 90,
        df["attendance_pct"] >= 75,
        df["attendance_pct"] >= 60
    ],
    [
        "Excellent",
        "Good",
        "Average"
    ],
    default="Poor"
)

category_count = df.groupby(
    "attendance_category"
).size()

print(category_count)


attendance_category
Excellent    42
Good         14
Poor          9
dtype: int64


In [16]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
#-----------------------------Write Solution Code-----------------------------

totals = df.agg({
    "total_classes": "sum",
    "present": "sum",
    "absent": "sum"
})

print("Total classes, present, and absent:")
print(totals)


Total classes, present, and absent:
total_classes    780
present          643
absent           137
dtype: int64


In [17]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""
#-----------------------------Write Solution Code-----------------------------

status_stats = df.groupby("status").agg(
    present_min=("present", "min"),
    present_max=("present", "max"),
    present_mean=("present", "mean"),
    absent_sum=("absent", "sum")
)

print(status_stats)


          present_min  present_max  present_mean  absent_sum
status                                                      
Critical            0            7      2.111111          89
Safe               11           12     11.619048          16
Warning             9           10      9.714286          32


### Section 4: Feature Engineering & Transformation

In [18]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""
#-----------------------------Write Solution Code-----------------------------

df["risk_score"] = (
    df["absent"] * 0.4
    + df["consecutive_absences"] * 0.6
)

top_10_risk = df.nlargest(10, "risk_score")

print(
    top_10_risk[
        ["roll_no", "student_name", "absent",
         "consecutive_absences", "risk_score"]
    ]
)


       roll_no     student_name  absent  consecutive_absences  risk_score
10  4MW24CS071       LAXMEESH V      12                    12        12.0
63  4MW25CS405  MANYA PRARTHANA      12                    12        12.0
5   4MW24CS066       KOUSHTUBHA      11                     5         7.4
35  4MW24CS096    NITHIL SHETTY      11                     5         7.4
46  4MW24CS107  PRATHAM G KUMAR      11                     5         7.4
49  4MW24CS110       PRATHWEESH      11                     5         7.4
13  4MW24CS074   MAHALINGAPPA C      11                     4         6.8
51  4MW24CS112            RAHUL       5                     2         3.2
29  4MW24CS090      NIKHIL NAIK       5                     0         2.0
33  4MW24CS094        NISHA M B       2                     1         1.4


In [19]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
#-----------------------------Write Solution Code-----------------------------

df["classes_needed_numeric"] = (
    df["classes_needed_75pct"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
)

df["classes_needed_numeric"] = (
    pd.to_numeric(
        df["classes_needed_numeric"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

print(
    df[
        ["classes_needed_75pct", "classes_needed_numeric"]
    ].head(15)
)


   classes_needed_75pct  classes_needed_numeric
0                 ✓ Met                       0
1                 ✓ Met                       0
2                 ✓ Met                       0
3                 ✓ Met                       0
4                 ✓ Met                       0
5               32 more                      32
6                 ✓ Met                       0
7                 ✓ Met                       0
8                 ✓ Met                       0
9                 ✓ Met                       0
10              36 more                      36
11                ✓ Met                       0
12                ✓ Met                       0
13              32 more                      32
14                ✓ Met                       0


In [20]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """
#-----------------------------Write Solution Code-----------------------------

df["attendance_flag"] = df["attendance_pct"].apply(
    lambda x: "Defaulter" if x < 75 else "Safe"
)

print(
    df[["student_name", "attendance_pct", "attendance_flag"]]
)


         student_name  attendance_pct attendance_flag
0        KAVYA KAMATH             100            Safe
1           KEERTHANA             100            Safe
2    KEERTHI.R.SHETTY             100            Safe
3              KHUSHI              92            Safe
4      KISHAN KUMAR K              92            Safe
..                ...             ...             ...
60           SAHANA M              83            Safe
61             SAKSHI              92            Safe
62  MANISH M BHANDARY              92            Safe
63    MANYA PRARTHANA               0       Defaulter
64              ROHAN              83            Safe

[65 rows x 3 columns]


In [21]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
#-----------------------------Write Solution Code-----------------------------

df["attendance_range"] = pd.cut(
    df["attendance_pct"],
    bins=[0, 50, 75, 90, 100],
    labels=["Critical", "Warning", "Good", "Excellent"],
    include_lowest=True
)

print(
    df[["student_name", "attendance_pct", "attendance_range"]]
)


         student_name  attendance_pct attendance_range
0        KAVYA KAMATH             100        Excellent
1           KEERTHANA             100        Excellent
2    KEERTHI.R.SHETTY             100        Excellent
3              KHUSHI              92        Excellent
4      KISHAN KUMAR K              92        Excellent
..                ...             ...              ...
60           SAHANA M              83             Good
61             SAKSHI              92        Excellent
62  MANISH M BHANDARY              92        Excellent
63    MANYA PRARTHANA               0         Critical
64              ROHAN              83             Good

[65 rows x 3 columns]


### Section 5: Advanced Analysis 

In [22]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
#-----------------------------Write Solution Code-----------------------------

pivot_summary = pd.pivot_table(
    df,
    index="status",
    columns="attendance_category",
    values="roll_no",
    aggfunc="count",
    fill_value=0
)

print(pivot_summary)


attendance_category  Excellent  Good  Poor
status                                    
Critical                     0     0     9
Safe                        42     0     0
Warning                      0    14     0


In [23]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
#-----------------------------Write Solution Code-----------------------------

# Extract the base name by removing trailing numbers
# or numbers enclosed in parentheses, if present
df["base_name"] = (
    df["student_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s*\(\d+\)\s*$", "", regex=True)
    .str.replace(r"\s+\d+\s*$", "", regex=True)
    .str.strip()
    .str.upper()
)

duplicate_names = df[
    df.duplicated(subset=["base_name"], keep=False)
].sort_values("base_name")

print("Duplicate student names:")
print(duplicate_names)


Duplicate student names:
Empty DataFrame
Columns: [roll_no, student_name, subject, total_classes, present, absent, excused, attendance_pct, status, classes_needed_75pct, last_present_date, consecutive_absences, attendance_category, risk_score, classes_needed_numeric, attendance_flag, attendance_range, base_name]
Index: []


In [24]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
#-----------------------------Write Solution Code-----------------------------

# Create a hypothetical Student_Details DataFrame
student_details = pd.DataFrame({
    "roll_no": df["roll_no"].drop_duplicates(),
    "branch": ["CSE"] * df["roll_no"].nunique(),
    "semester": [5] * df["roll_no"].nunique()
})

# Perform an inner join
merged_df = pd.merge(
    df,
    student_details,
    on="roll_no",
    how="inner"
)

print("Merged DataFrame:")
print(merged_df)


Merged DataFrame:
       roll_no       student_name                                  subject  \
0   4MW24CS061       KAVYA KAMATH  Data Science & GenAI - 3nd Yr - CSE - B   
1   4MW24CS062          KEERTHANA  Data Science & GenAI - 3nd Yr - CSE - B   
2   4MW24CS063   KEERTHI.R.SHETTY  Data Science & GenAI - 3nd Yr - CSE - B   
3   4MW24CS064             KHUSHI  Data Science & GenAI - 3nd Yr - CSE - B   
4   4MW24CS065     KISHAN KUMAR K  Data Science & GenAI - 3nd Yr - CSE - B   
..         ...                ...                                      ...   
60  4MW24CS121           SAHANA M  Data Science & GenAI - 3nd Yr - CSE - B   
61  4MW24CS122             SAKSHI  Data Science & GenAI - 3nd Yr - CSE - B   
62  4MW25CS404  MANISH M BHANDARY  Data Science & GenAI - 3nd Yr - CSE - B   
63  4MW25CS405    MANYA PRARTHANA  Data Science & GenAI - 3nd Yr - CSE - B   
64  4MW25CS406              ROHAN  Data Science & GenAI - 3nd Yr - CSE - B   

    total_classes  present  absent  excused  

In [25]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""
#-----------------------------Write Solution Code-----------------------------

# (a) Export all Critical students
critical_students = df[df["status"] == "Critical"]
critical_students.to_csv(
    "critical_students.csv",
    index=False
)

# (b) Export summary statistics grouped by Status
status_summary = df.groupby("status").agg(
    student_count=("roll_no", "count"),
    average_attendance=("attendance_pct", "mean"),
    maximum_consecutive_absences=("consecutive_absences", "max"),
    total_present=("present", "sum"),
    total_absent=("absent", "sum")
)

status_summary.to_csv("status_summary.csv")

# (c) Export top 10 students by Risk Score
top_10_risk = df.nlargest(10, "risk_score")

top_10_risk.to_csv(
    "top_10_risk_students.csv",
    index=False
)

print("All three CSV files exported successfully!")


All three CSV files exported successfully!
